#### Faiss

Facebook Al Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader('speech.txt')
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=500,chunk_overlap=30)
docs=text_splitter.split_documents(documents)

Created a chunk of size 599, which is longer than the specified 500
Created a chunk of size 617, which is longer than the specified 500


In [7]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a language's lexicon. There are many different intentional speech acts, such as informing, declaring, asking, persuading, directing; acts may vary in various aspects like enunciation, intonation, loudness, and tempo to convey meaning. Individuals may also unintentionally communicate aspects of their social position through speech, such as sex, age, place of origin, physiological and mental condition, education, and experiences."),
 Document(metadata={'source': 'speech.txt'}, page_content="While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Le

In [9]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")
db=FAISS.from_documents(docs,embeddings)
db

In [12]:
#querying
query="While normally used to facilitate communication with others, people may also use speech without the intent to communicate."
docs=db.similarity_search(query)
docs[0].page_content

"While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Lev Vygotsky) have maintained is the use of silent speech in an interior monologue to vivify and organize cognition, sometimes in the momentary adoption of a dual persona as self addressing self as though addressing another person. Solo speech can be used to memorize or to test one's memorization of things, and in prayer or in meditation."

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [14]:
#retriever
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

"While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Lev Vygotsky) have maintained is the use of silent speech in an interior monologue to vivify and organize cognition, sometimes in the momentary adoption of a dual persona as self addressing self as though addressing another person. Solo speech can be used to memorize or to test one's memorization of things, and in prayer or in meditation."

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [16]:
#here, since we're using this similarity_search_with_score, we are getting L2 score/manhattan score for each of them with the output as you can see in the output!!!
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='eb631e5e-bdb5-4e3e-bb82-14f2fa809380', metadata={'source': 'speech.txt'}, page_content="While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Lev Vygotsky) have maintained is the use of silent speech in an interior monologue to vivify and organize cognition, sometimes in the momentary adoption of a dual persona as self addressing self as though addressing another person. Solo speech can be used to memorize or to test one's memorization of things, and in prayer or in meditation."),
  np.float32(226.32773)),
 (Document(id='7e5cebab-31e7-414d-852c-a2c57c6c7826', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like wo

In [17]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[0.36015620827674866,
 0.9504265785217285,
 -3.0363028049468994,
 -1.0385162830352783,
 1.2524479627609253,
 1.1425212621688843,
 -0.12571267783641815,
 0.17737053334712982,
 0.20880715548992157,
 -0.258215069770813,
 -0.4533765912055969,
 0.5320417284965515,
 0.15081992745399475,
 1.6666514873504639,
 0.8099531531333923,
 -0.4355597198009491,
 0.3969501256942749,
 -1.2759748697280884,
 -1.244185447692871,
 1.9332971572875977,
 -2.0218346118927,
 -0.39272603392601013,
 0.08812081813812256,
 -0.34447911381721497,
 0.4235643446445465,
 0.39533531665802,
 0.19526506960391998,
 -0.09674083441495895,
 -0.8591043949127197,
 -0.4451022446155548,
 0.6161379218101501,
 0.024600915610790253,
 0.04314842075109482,
 0.3288497030735016,
 -1.5328091382980347,
 -1.264495849609375,
 -0.1875298023223877,
 0.6993769407272339,
 -0.11204288899898529,
 0.09613793343305588,
 0.60265052318573,
 0.7240580916404724,
 -0.4434041380882263,
 -0.9375011324882507,
 1.4728732109069824,
 -0.3381149172782898,
 0.98151

In [19]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='eb631e5e-bdb5-4e3e-bb82-14f2fa809380', metadata={'source': 'speech.txt'}, page_content="While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Lev Vygotsky) have maintained is the use of silent speech in an interior monologue to vivify and organize cognition, sometimes in the momentary adoption of a dual persona as self addressing self as though addressing another person. Solo speech can be used to memorize or to test one's memorization of things, and in prayer or in meditation."),
 Document(id='7e5cebab-31e7-414d-852c-a2c57c6c7826', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a langu

In [20]:
#Saving and Loading
db.save_local("faiss_index")

In [23]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)
docs

[Document(id='eb631e5e-bdb5-4e3e-bb82-14f2fa809380', metadata={'source': 'speech.txt'}, page_content="While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Lev Vygotsky) have maintained is the use of silent speech in an interior monologue to vivify and organize cognition, sometimes in the momentary adoption of a dual persona as self addressing self as though addressing another person. Solo speech can be used to memorize or to test one's memorization of things, and in prayer or in meditation."),
 Document(id='7e5cebab-31e7-414d-852c-a2c57c6c7826', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a langu